# D33 Step 9 + 选路先验诊断(服务器 72 核版)

在校服务器上跑:
1. **Step 9 单点扰动扫描**(① reroute 延误级联 / ② wait 时机轴),点级并行 + 断点重续;
2. **选路先验闭环对照**(Step 10 前诊断):典型路查表(BC 代理)vs 信号员,看纯典型路能不能≈信号员。

覆盖 **train+val = 75 天**(D40,test 不碰)。每格从上往下跑;中断后重跑同一格即续跑。

## 设置(定位项目根 + 核数)

In [ ]:
import os, sys
p = os.getcwd()
while p != os.path.dirname(p) and not os.path.isdir(os.path.join(p, 'derbysim')):
    p = os.path.dirname(p)
assert os.path.isdir(os.path.join(p, 'derbysim')), '找不到项目根(含 derbysim/ 的目录)'
os.chdir(p); sys.path.insert(0, p)
os.environ.setdefault('PYTHONPYCACHEPREFIX', '/tmp/pyc_step9')
N_WORKERS = 70   # 72 核留点余量; 内存 4GB/核充足。可改 72。
print('project root:', p, '| cpu:', os.cpu_count(), '| N_WORKERS:', N_WORKERS)

In [ ]:
# 缺依赖时取消注释运行(tqdm 进度条 / pyarrow 读 parquet)
# !pip install tqdm pyarrow --quiet

---
## A) 选路先验闭环对照 —— Step 10 前诊断(先跑这个,~15–20 分钟)

典型路查表(BC 代理,纯走常走路)vs 信号员 vs greedy vs random,看纯典型路的完成率/延误是不是≈信号员。
先 val(快),再 train+val(D40 全量)。跑完把打印的 `wrote ...json` 文件名交回。

In [ ]:
!python scripts/24_routing_prior_closed_loop.py --split val

In [ ]:
!python scripts/24_routing_prior_closed_loop.py --split trainval

---
## B) Step 9 单点扰动扫描(72 核并行,断点重续)

> 结果已跑过就可跳过;要重头来删掉对应 `artifacts/reports/d33_step9_<mode>_scan.jsonl` 再跑。

### 0) Smoke(1 天 8 点,单独文件,不污染正式结果)

In [ ]:
from derbysim.learn.step9_parallel import run_scan_parallel, summarize_scan
run_scan_parallel('reroute', split='val', n_workers=min(8, N_WORKERS),
                  limit_days=1, limit_points=8,
                  out='artifacts/reports/_smoke_step9.jsonl')

### 1) 全量 ① reroute(延误级联,~31k 点)。中断后重跑本格即续跑。

In [ ]:
run_scan_parallel('reroute', split='trainval', n_workers=N_WORKERS)

In [ ]:
summarize_scan('reroute')   # 写 d33_step9_reroute_summary_*.json 并打印

### 2) 全量 ② wait(时机轴,~42k 点)

In [ ]:
run_scan_parallel('wait', split='trainval', n_workers=N_WORKERS)

In [ ]:
summarize_scan('wait')

### 说明
- 选路对照:`artifacts/reports/routing_prior_closedloop_<split>_*.json`。
- Step 9 逐点:`artifacts/reports/d33_step9_<mode>_scan.jsonl`(可断点重续);汇总:`..._summary_*.json`。
- 把相应文件名(或内容)交回即可继续分析。